# Notebook 2: Train the source-only baseline (no adaptation)

**What we do:** fine-tune an ImageNet-pretrained ResNet-50 on **PlantVillage only**, then see how it behaves on
PlantDoc. This is the "no adaptation" row of our comparison and the starting point for every other method.

**Why it matters:** this is where we first *see* the reliability gap with our own eyes: accuracy should drop a
lot on field images while the model's confidence stays high.

### Where this notebook fits
| Notebook | Job |
|---|---|
| 01 (done) | Decide which image has which role, save `master_manifest.csv` |
| **02 (this one)** | **Train the source-only model, save its outputs** |
| 03 | Evaluate: accuracy, ECE, Brier, temperature scaling, selective risk, for Setting A and B |
| 04 | Baselines: AdaBN, moment matching, DANN |
| 05 | Deep alignment: Deep CORAL, JMMD |
| 06 | Optimal transport: JDOT, DeepJDOT |
| 07 | Final comparison, statistics, Grad-CAM |

### The key design idea
Training is slow (needs a GPU), analysis is fast. So this notebook **saves the model's raw outputs to a file**
(the *logits*, which are the scores before softmax, and the 2048-number *features* before the final layer).
Every later method will save its outputs in the **same format**, and notebook 03 will evaluate all of them
with the same code. That keeps the comparison fair and means we never retrain just to compute a new metric.

### Before the full run: do a DEBUG run
`DEBUG = True` uses a few hundred images and one epoch, so the whole notebook finishes in minutes, even on a
CPU. Use it to catch mistakes cheaply. Switch it to `False` only when everything runs cleanly.

The full run needs a **GPU** (for example a local one, Google Colab or Kaggle). On a CPU it would take far too long.

## Step 1: Imports and settings

In [2]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

In [3]:
# ---- Edit these -------------------------------------------------------------
DEBUG = False                     # True = tiny quick run to test the code. False = the real experiment.

MANIFEST = Path("outputs/master_manifest.csv")   # made by notebook 01
PV_ROOT = Path(r"F:\mphil\pv")              # same folders as in notebook 01
PD_ROOT = Path(r"F:\mphil\pdoc")
OUT_DIR = Path("outputs")

TRAIN_SEED = 42                    # random seed for training (later we repeat with several seeds)  changed to 42 from 0

# Training settings (follow the reference study where it states them: AdamW, lr 1e-4, cosine schedule,
# 20 epochs, early stopping. Weight decay and patience are our own choices.)
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5                      # stop if validation accuracy has not improved for this many epochs
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0                  # parallel image loading; set 0 if you get errors on Windows
# -----------------------------------------------------------------------------

if DEBUG:
    EPOCHS = 1
    print("*** DEBUG MODE: EPOCHS forced to 1 and only ~128 images per group are used. ***")
    print("*** Set DEBUG = False and restart the kernel for the real experiment.      ***")

(OUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| DEBUG =", DEBUG)


def set_seed(seed):
    # Makes the 'random' parts (weight init, shuffling, augmentation) repeatable
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(TRAIN_SEED)

Using device: cuda | DEBUG = False


## Step 2: Load the manifest from notebook 01

The manifest tells us the role of every image. We add the full file path and a numeric **label** for each class
(the network needs numbers, not names). PlantDoc's *unknown* classes get label `-1` because the model has no
output for them.

In [4]:
manifest = pd.read_csv(MANIFEST, keep_default_na=False)

# Full path = dataset root + the relative path saved in the manifest
roots = {"pv": PV_ROOT, "pd": PD_ROOT}
manifest["full_path"] = [str(roots[d] / p) for d, p in zip(manifest["dataset"], manifest["path"])]

# One number per known class (alphabetical order, so it is the same in every notebook)
known_roles = ["pv_train", "pv_val", "pv_test", "pd_known"]
classes = sorted(manifest.loc[manifest["role"].isin(known_roles), "class"].unique())
class_to_id = {name: i for i, name in enumerate(classes)}
manifest["label"] = manifest["class"].map(class_to_id).fillna(-1).astype(int)

print(len(classes), "classes")
manifest["role"].value_counts()

21 classes


role
pv_train      18601
pv_test        3987
pv_val         3986
pd_known       2018
pd_unknown      466
Name: count, dtype: int64

In [5]:
# Split the manifest into the five groups we will feed to the network
group_names = ["pv_train", "pv_val", "pv_test", "pd_known", "pd_unknown"]
groups = {name: manifest[manifest["role"] == name].reset_index(drop=True) for name in group_names}

if DEBUG:
    # Keep only a handful of images per group so the test run is fast
    for name in group_names:
        groups[name] = groups[name].sample(min(128, len(groups[name])), random_state=0).reset_index(drop=True)

# Check that the files really exist before we start (catches wrong paths immediately)
missing = [p for p in groups["pv_train"]["full_path"].head(50) if not Path(p).exists()]
assert not missing, f"Image files not found, check PV_ROOT / PD_ROOT. Example: {missing[0]}"

for name in group_names:
    print(f"{name:11s}: {len(groups[name])} images")

pv_train   : 18601 images
pv_val     : 3986 images
pv_test    : 3987 images
pd_known   : 2018 images
pd_unknown : 466 images


## Step 3: Turn files into network inputs

* **Training images** get light random changes (crop, flip, colour jitter) so the model doesn't just memorise.
* **Evaluation images** are only resized and centre-cropped, so results are repeatable.
* Both are normalised with the ImageNet averages, because the pretrained ResNet expects that.

In [6]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    normalize,
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

In [7]:
# A Dataset tells PyTorch how to fetch image number i together with its label
class ImageTable(Dataset):
    def __init__(self, table, transform):
        self.paths = table["full_path"].tolist()
        self.labels = table["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert("RGB")
        return self.transform(image), self.labels[i]


pin = device.type == "cuda"

train_loader = DataLoader(ImageTable(groups["pv_train"], train_transform), batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=pin)

# Evaluation loaders: shuffle=False keeps the SAME order as the tables, which we rely on when saving outputs
eval_loaders = {name: DataLoader(ImageTable(groups[name], eval_transform), batch_size=64,
                                 shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
                for name in ["pv_val", "pv_test", "pd_known", "pd_unknown"]}

## Step 4: The model

ResNet-50 pretrained on ImageNet (it downloads the weights the first time, so you need an internet connection).
We replace only its final layer with one that has 21 outputs, one per class. Everything else is kept and then
fine-tuned.

We also create `body`: the same network **without** its final layer. It outputs the 2048 *features* that the final
layer uses. `body` and `model` share the same weights; `body` just lets us read the features out. Later
methods (AdaBN, moment matching, JDOT, ...) work on exactly these features.

In [8]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(classes))      # new final layer: 2048 -> 21 classes
model = model.to(device)

# Everything except the last layer, sharing weights with `model`
body = nn.Sequential(*list(model.children())[:-1])

n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50 with {len(classes)} outputs, {n_params / 1e6:.1f} million parameters")

ResNet-50 with 21 outputs, 23.6 million parameters


## Step 5: Helper functions

Two small tools:
* `predict` runs a loader through the model and returns the **logits** (raw scores), the **features**, and the labels.
* `accuracy` compares the top-scoring class with the true label.

In [9]:
@torch.no_grad()                      # we are not learning here, so no gradients are needed
def predict(loader):
    model.eval()                      # switches off augmentation-style randomness (dropout, batch-norm updates)
    all_logits, all_features, all_labels = [], [], []
    for images, labels in loader:
        images = images.to(device)
        features = body(images).flatten(1)            # (batch, 2048)
        logits = model.fc(features)                   # (batch, 21)
        all_logits.append(logits.cpu().numpy())
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_logits), np.concatenate(all_features), np.concatenate(all_labels)


def accuracy(logits, labels):
    return float((logits.argmax(axis=1) == labels).mean())

## Step 6: Train

Each **epoch** is one pass over all PlantVillage training images. After each epoch we check accuracy on the
PlantVillage **validation** set. We keep the best model, and we stop early if it hasn't improved for
`PATIENCE` epochs. (Notice we never look at PlantDoc here: this model has seen only the lab domain.)

* **Loss** (cross-entropy): how wrong the predicted probabilities are.
* **AdamW**: the optimiser that updates the weights.
* **Cosine schedule**: the learning rate slowly shrinks over training.

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Debug runs get their own file name, so they can never overwrite the real trained weights
checkpoint_path = OUT_DIR / "checkpoints" / f"source_only_seed{TRAIN_SEED}{'_DEBUG' if DEBUG else ''}.pt"
best_val_acc, epochs_without_improvement = 0.0, 0
epochs_completed = 0               # how many epochs actually ran (saved with the outputs)

In [11]:
for epoch in range(1, EPOCHS + 1):
    # ---- learn from the training images ----
    model.train()
    running_loss, n_seen = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                 # clear the previous step's gradients
        loss = criterion(model(images), labels)
        loss.backward()                       # work out how to change each weight
        optimizer.step()                      # change the weights a little
        running_loss += loss.item() * len(labels)
        n_seen += len(labels)
    scheduler.step()

    # ---- check on validation images ----
    val_logits, _, val_labels = predict(eval_loaders["pv_val"])
    val_acc = accuracy(val_logits, val_labels)
    print(f"epoch {epoch:2d} | train loss {running_loss / n_seen:.4f} | PlantVillage val accuracy {val_acc:.4f}")
    epochs_completed = epoch

    # ---- keep the best model, stop early if it stops improving ----
    if val_acc > best_val_acc:
        best_val_acc, epochs_without_improvement = val_acc, 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping: no improvement for", PATIENCE, "epochs")
            break

print("Best validation accuracy:", round(best_val_acc, 4))

epoch  1 | train loss 0.2393 | PlantVillage val accuracy 0.9742
epoch  2 | train loss 0.0681 | PlantVillage val accuracy 0.9812
epoch  3 | train loss 0.0496 | PlantVillage val accuracy 0.9870
epoch  4 | train loss 0.0362 | PlantVillage val accuracy 0.9887
epoch  5 | train loss 0.0300 | PlantVillage val accuracy 0.9834
epoch  6 | train loss 0.0215 | PlantVillage val accuracy 0.9922
epoch  7 | train loss 0.0251 | PlantVillage val accuracy 0.9877
epoch  8 | train loss 0.0229 | PlantVillage val accuracy 0.9920
epoch  9 | train loss 0.0167 | PlantVillage val accuracy 0.9942
epoch 10 | train loss 0.0094 | PlantVillage val accuracy 0.9940
epoch 11 | train loss 0.0099 | PlantVillage val accuracy 0.9940
epoch 12 | train loss 0.0066 | PlantVillage val accuracy 0.9960
epoch 13 | train loss 0.0053 | PlantVillage val accuracy 0.9957
epoch 14 | train loss 0.0028 | PlantVillage val accuracy 0.9962
epoch 15 | train loss 0.0020 | PlantVillage val accuracy 0.9937
epoch 16 | train loss 0.0018 | PlantVill

## Step 7: Run the best model on every group and save the outputs

We reload the best saved weights, then collect logits and features for the validation, test, PlantDoc-known and
PlantDoc-unknown groups. The `paths` we save let notebook 03 match each output back to its row in the manifest,
and therefore to Setting A or B.

In [12]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

results = {}
for name, loader in eval_loaders.items():
    logits, features, labels = predict(loader)
    results[name] = {"logits": logits, "features": features, "labels": labels,
                     "paths": groups[name]["path"].to_numpy()}
    print(f"{name:11s}: logits {logits.shape}, features {features.shape}")

pv_val     : logits (3986, 21), features (3986, 2048)
pv_test    : logits (3987, 21), features (3987, 2048)
pd_known   : logits (2018, 21), features (2018, 2048)
pd_unknown : logits (466, 21), features (466, 2048)


In [14]:
save_dict = {"classes": np.array(classes),
             "epochs_run": np.array(epochs_completed),     # recorded so the file documents how it was made
             "debug_mode": np.array(DEBUG),
             # the final layer, so later notebooks can re-classify adapted features without PyTorch
             "fc_weight": model.fc.weight.detach().cpu().numpy(),
             "fc_bias": model.fc.bias.detach().cpu().numpy()}
for name, r in results.items():
    save_dict[f"{name}_logits"] = r["logits"].astype(np.float32)
    save_dict[f"{name}_features"] = r["features"].astype(np.float16)   # half precision saves disk space
    save_dict[f"{name}_labels"] = r["labels"]
    save_dict[f"{name}_paths"] = r["paths"]

output_file = OUT_DIR / f"outputs_source_only_seed{TRAIN_SEED}{'_DEBUG' if DEBUG else ''}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file, f"({output_file.stat().st_size / 1e6:.1f} MB)")

Saved outputs\outputs_source_only_seed42.npz (39.6 MB)


## Step 8: A first look at the reliability gap

Two numbers per group:
* **accuracy**: how often the top prediction is right
* **mean confidence**: the average of the model's highest predicted probability (`softmax` turns scores into
  probabilities that sum to 1)

A *well-behaved* model has confidence close to accuracy. If accuracy collapses but confidence doesn't,
that is exactly the problem our project studies.

In [13]:
def softmax(scores):
    scores = scores - scores.max(axis=1, keepdims=True)     # subtract the max for numerical safety
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)


rows = []
for name in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    probs = softmax(results[name]["logits"])
    row = {"group": name, "images": len(probs), "mean_confidence": round(float(probs.max(axis=1).mean()), 4)}
    if name != "pd_unknown":     # unknown images have no correct class among our 21
        row["accuracy"] = round(accuracy(results[name]["logits"], results[name]["labels"]), 4)
    rows.append(row)

pd.DataFrame(rows).set_index("group")

,images,mean_confidence,accuracy
group,,,
pv_val,3986,0.9980,0.9967
pv_test,3987,0.9976,0.9947
pd_known,2018,0.8382,0.2374
pd_unknown,466,0.7651,NaN


### What to expect (full run, `DEBUG = False`)
The reference study reports about **99.7%** accuracy on PlantVillage, about **32%** on PlantDoc (this is Setting
A's test set), and a mean confidence of about **0.80** on PlantDoc. Our numbers will differ a little because our
splits and seeds are not identical, but if you see something like 99% / 30-40% / 0.7-0.85, the pipeline is behaving.
If PlantDoc accuracy comes out near 90%, something is wrong (for example PlantDoc images leaking into training).

*In DEBUG mode the numbers mean nothing (one epoch on a few images). It only proves the code runs.*

### Next
Notebook 03 loads the saved `.npz` and computes the full reliability metrics, including temperature scaling, for
Setting A and Setting B.